In [ ]:

from pdb import pm
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


# thay các đường dẫn thư mục
# pm_path = r"D:\mttq\openaq_pm25_hanoi_hcmc.csv"
# wx_path = r"D:\mttq\openmeteo_weather_hanoi_hcmc.csv"
# out_dir = Path(r"D:\mttq\output")
# out_dir.mkdir(parents=True, exist_ok=True)

# pm = pd.read_csv(r"D:\mttq\openaq_pm25_hanoi_hcmc.csv", parse_dates=["event_time"])
# wx = pd.read_csv(r"D:\mttq\openmeteo_weather_hanoi_hcmc.csv", parse_dates=["event_time"])


df = pm[["city", "station_id", "station_name", "event_time", "pm25_ug_m3"]].merge(
    wx.drop(columns=["city", "station_name", "latitude", "longitude"]),
    on=["station_id", "event_time"],
    how="inner",
)

# Hướng gió là biến vòng (0° = 360°) -> tách thành thành phần gió Bắc/Đông
rad = np.deg2rad(df["wind_direction_deg"])
df["wind_north"] = -df["wind_speed_m_s"] * np.cos(rad)  # >0: gió thổi về phía Bắc
df["wind_east"] = -df["wind_speed_m_s"] * np.sin(rad)   # >0: gió thổi về phía Đông
df["hour"] = df["event_time"].dt.tz_convert("Asia/Ho_Chi_Minh").dt.hour

rename = {
    "pm25_ug_m3": "PM2.5",
    "temperature_c": "Nhiệt độ",
    "relative_humidity_pct": "Độ ẩm",
    "wind_speed_m_s": "Tốc độ gió",
    "wind_north": "Gió Bắc",
    "wind_east": "Gió Đông",
    "precipitation_mm": "Lượng mưa",
}
cols = list(rename.values())
data = df.rename(columns=rename)

print(f"Số dòng sau khi ghép: {len(data)} "
      f"({data.groupby('city').size().to_dict()})")

# ---------- 3. Thống kê mô tả ----------
desc = data.groupby("city")[cols].agg(["mean", "std", "min", "max"]).round(2)
desc.to_csv(out_dir / "descriptive_stats.csv")
print(desc.T.to_string())

# ---------- 4. Ma trận tương quan ----------
def corr_heatmap(ax, frame, title, method="pearson"):
    c = frame[cols].corr(method=method)
    mask = np.triu(np.ones_like(c, dtype=bool), k=1)
    sns.heatmap(c, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
                vmin=-1, vmax=1, center=0, square=True, linewidths=.5,
                cbar_kws={"shrink": .75, "label": "hệ số tương quan"}, ax=ax)
    ax.set_title(f"{title} (n={len(frame)})", fontsize=12)
    ax.tick_params(axis="x", rotation=40)
    return c

groups = {
    "Hà Nội": data[data.city == "Hanoi"],
    "TP.HCM": data[data.city == "HCMC"],
    "Cả hai thành phố": data,
}

for method, label in [("pearson", "Pearson"), ("spearman", "Spearman")]:
    fig, axes = plt.subplots(1, 3, figsize=(21, 6.5))
    for ax, (name, frame) in zip(axes, groups.items()):
        c = corr_heatmap(ax, frame, f"{name} - {label}", method)
        c.round(3).to_csv(out_dir / f"corr_{method}_{name.replace(' ', '_').replace('.', '')}.csv")
    fig.suptitle(f"Ma trận tương quan {label}: PM2.5 và thời tiết, 15-21/11/2025 (theo giờ)",
                 fontsize=14, y=1.02)
    fig.tight_layout()
    fig.savefig(out_dir / f"correlation_matrix_{method}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

# ---------- 5. Tương quan của PM2.5 với từng biến, xếp theo độ mạnh ----------
print("\nTương quan Pearson với PM2.5:")
for name, frame in groups.items():
    s = frame[cols].corr()["PM2.5"].drop("PM2.5").sort_values(key=abs, ascending=False)
    print(f"\n[{name}]\n{s.round(3).to_string()}")

# ---------- 6. Tương quan theo từng trạm Hà Nội (kiểm tra độ ổn định) ----------
print("\nTương quan PM2.5 - từng trạm:")
per_station = (data.groupby(["city", "station_name"])[cols].corr()["PM2.5"]
               .unstack().drop(columns="PM2.5").round(2))
print(per_station.to_string())
per_station.to_csv(out_dir / "corr_per_station.csv")

data.to_csv(out_dir / "merged_pm25_weather.csv", index=False)

Số dòng sau khi ghép: 815 ({'HCMC': 164, 'Hanoi': 651})
city               HCMC   Hanoi
PM2.5      mean   35.05   43.13
           std    22.24   29.90
           min     7.79    0.47
           max   136.00  170.00
Nhiệt độ   mean   25.60   19.08
           std     2.09    3.78
           min    21.80   12.60
           max    31.30   27.30
Độ ẩm      mean   85.35   75.56
           std    11.01   13.37
           min    51.00   39.00
           max   100.00   96.00
Tốc độ gió mean    1.81    2.79
           std     0.80    1.42
           min     0.10    0.11
           max     4.25    7.03
Gió Bắc    mean   -1.26   -2.58
           std     1.13    1.37
           min    -4.24   -6.10
           max     1.45    0.25
Gió Đông   mean    0.14   -0.06
           std     1.02    1.13
           min    -3.56   -4.53
           max     2.10    1.76
Lượng mưa  mean    0.40    0.12
           std     1.24    0.29
           min     0.00    0.00
           max    10.00    1.70

Tương quan Pear